# Experimento B — Expansión del dataset de entrenamiento

Este notebook analiza el efecto de expandir el dataset de entrenamiento del DCGAN  
de **173 fragmentos** originales a **2518 ventanas** generadas mediante un parser VGLC  
con sliding window (stride 4) sobre todos los niveles de SMB disponibles.

**Hipótesis:** más datos de entrenamiento deben mejorar la calidad estructural de los  
niveles generados, especialmente en `pipe_completeness` y `ground_continuity`.

**Resultado anticipado:** la hipótesis se confirma *parcialmente* — exploraremos por qué  
la expansión no es suficiente por sí sola.

## 1. Setup — Imports y configuración de paths

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ── Paths relativos desde notebooks/ ──────────────────────────────────────────
NOTEBOOK_DIR  = os.path.abspath('')
PROJECT_ROOT  = os.path.abspath('../../')
DAGSTUHL_PATH = os.path.join(PROJECT_ROOT, 'clone', 'DagstuhlGAN', 'pytorch')
SRC_PATH      = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'src')
BASELINE_DIR  = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'experiments', 'baseline')
EXPB_DIR      = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'experiments', 'expanded_data')
DATA_DIR      = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'data')

for p in [DAGSTUHL_PATH, SRC_PATH]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Python :', sys.version)
print('PROJECT_ROOT :', PROJECT_ROOT)
print('DAGSTUHL_PATH existe:', os.path.isdir(DAGSTUHL_PATH))
print('EXPB_DIR existe     :', os.path.isdir(EXPB_DIR))

## 2. Comparación del tamaño de dataset

Comparamos los dos datasets usados para entrenar el DCGAN:

- **Dataset original** (`example_original.json`): 173 fragmentos del repositorio Dagstuhl.
- **Dataset expandido** (`dataset_full.json`): ventanas 14×28 con stride=4 sobre todos los niveles VGLC.

In [ ]:
# ── Cargar ambos datasets ─────────────────────────────────────────────────────
orig_path = os.path.join(DAGSTUHL_PATH, 'example_original.json')
full_path = os.path.join(DATA_DIR, 'dataset_full.json')

datasets = {}

if os.path.exists(orig_path):
    with open(orig_path, 'r') as f:
        orig_data = json.load(f)
    orig_arr = np.array(orig_data)
    datasets['original'] = orig_arr
    print(f'Dataset original : {orig_arr.shape}  —  {len(orig_data)} fragmentos')
else:
    print(f'No encontrado: {orig_path}')
    datasets['original'] = None

if os.path.exists(full_path):
    with open(full_path, 'r') as f:
        full_data = json.load(f)
    full_arr = np.array(full_data)
    datasets['expanded'] = full_arr
    print(f'Dataset expandido: {full_arr.shape}  —  {len(full_data)} ventanas')
else:
    print(f'No encontrado: {full_path}')
    datasets['expanded'] = None

In [ ]:
# ── Tabla comparativa de datasets ─────────────────────────────────────────────
n_orig = len(datasets['original']) if datasets['original'] is not None else 173
n_full = len(datasets['expanded']) if datasets['expanded'] is not None else 2518

print('\n' + '=' * 65)
print(f"{'Característica':<30} {'Dataset A (orig)':>16} {'Dataset B (exp)':>16}")
print('=' * 65)

rows = [
    ('Fragmentos / ventanas',     f'{n_orig}',            f'{n_full}'),
    ('Dimensión de cada muestra', '14 × 28',              '14 × 28'),
    ('Fuente',                    'Dagstuhl repo',        'VGLC parser'),
    ('Stride de ventana',         'N/A (manual)',         '4 tiles'),
    ('Cobertura de niveles',      '~5 niveles SMB',       'Todos los niveles SMB'),
    ('Factor de aumento',         '1×',                   f'{n_full / n_orig:.1f}×'),
]

for label, v_a, v_b in rows:
    print(f"  {label:<28} {v_a:>16} {v_b:>16}")

print('=' * 65)

# ── Gráfico de barras comparativo ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.5))
labels = ['Dataset A\n(original)', 'Dataset B\n(expandido)']
counts = [n_orig, n_full]
colors = ['#3498db', '#e67e22']
bars   = ax.bar(labels, counts, color=colors, edgecolor='#222', width=0.45)

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f'{count:,}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Número de fragmentos de entrenamiento')
ax.set_title('Tamaño del dataset: Exp A vs Exp B', fontsize=13, fontweight='bold')
ax.set_ylim(0, n_full * 1.2)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f'\nEl dataset B es {n_full/n_orig:.1f}x mayor que el dataset A.')

## 3. Distribución de tiles en ambos datasets

In [ ]:
TILE_NAMES = {
    0: 'Suelo', 1: 'Breakable', 2: 'Vacío', 3: 'Q-lleno',
    4: 'Q-vacío', 5: 'Enemigo', 6: 'Pipe TL', 7: 'Pipe TR',
    8: 'Pipe L',  9: 'Pipe R',
}

def tile_distribution(arr, label):
    flat   = arr.flatten()
    total  = len(flat)
    print(f'\n── {label} ──')
    dist   = {}
    for tile_id in range(10):
        count = int(np.sum(flat == tile_id))
        pct   = 100.0 * count / total
        dist[tile_id] = pct
        if count > 0:
            name = TILE_NAMES.get(tile_id, f'Tile {tile_id}')
            print(f'  [{tile_id}] {name:<12}: {pct:5.2f}%')
    return dist

dist_orig = None
dist_full = None

if datasets['original'] is not None:
    dist_orig = tile_distribution(datasets['original'], 'Dataset A (original)')
if datasets['expanded'] is not None:
    dist_full = tile_distribution(datasets['expanded'], 'Dataset B (expandido)')

# ── Gráfico comparativo de distribución ───────────────────────────────────────
if dist_orig is not None and dist_full is not None:
    tile_ids = sorted(set(list(dist_orig.keys()) + list(dist_full.keys())))
    tile_ids = [t for t in tile_ids if dist_orig.get(t, 0) > 0 or dist_full.get(t, 0) > 0]
    tile_labels = [TILE_NAMES.get(t, f'T{t}') for t in tile_ids]

    x    = np.arange(len(tile_ids))
    w    = 0.35
    vals_a = [dist_orig.get(t, 0) for t in tile_ids]
    vals_b = [dist_full.get(t, 0) for t in tile_ids]

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(x - w/2, vals_a, w, label='Dataset A (orig)', color='#3498db', edgecolor='#222')
    ax.bar(x + w/2, vals_b, w, label='Dataset B (exp)',  color='#e67e22', edgecolor='#222')
    ax.set_xticks(x)
    ax.set_xticklabels(tile_labels, rotation=25, ha='right', fontsize=9)
    ax.set_ylabel('Porcentaje de tiles (%)')
    ax.set_title('Distribución de tiles: Dataset A vs B', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

## 4. Curvas de entrenamiento (Exp B)

In [ ]:
progress_path = os.path.join(EXPB_DIR, 'training_progress.png')

if os.path.exists(progress_path):
    img = mpimg.imread(progress_path)
    h, w = img.shape[:2]
    fig_w = min(16, w / 80)
    fig_h = fig_w * h / w
    fig, ax = plt.subplots(figsize=(max(fig_w, 12), max(fig_h, 4)))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Curvas de entrenamiento — Exp B (dataset expandido)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print(f'Imagen no encontrada: {progress_path}')

# ── Training log (si existe) ──────────────────────────────────────────────────
log_path = os.path.join(EXPB_DIR, 'training_log.txt')
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        lines = f.readlines()
    print(f'\nTraining log ({len(lines)} líneas) — últimas 10:')
    for line in lines[-10:]:
        print(' ', line.rstrip())

## 5. Comparación de métricas: Exp A vs Exp B

In [ ]:
# ── Cargar métricas de ambos experimentos ─────────────────────────────────────
with open(os.path.join(BASELINE_DIR, 'metrics_baseline.json'), 'r') as f:
    data_a = json.load(f)

with open(os.path.join(EXPB_DIR, 'metrics_baseline.json'), 'r') as f:
    data_b = json.load(f)

m_a = data_a['metrics']
m_b = data_b['metrics']

METRIC_LABELS = {
    'pipe_completeness' : 'Completitud de tuberías',
    'ground_continuity' : 'Continuidad de suelo',
    'gap_traversability': 'Traversabilidad de gaps',
    'enemy_placement'   : 'Colocación de enemigos',
    'structural_avg'    : 'Promedio estructural',
}

print('\n' + '=' * 75)
print(f"{'Métrica':<35} {'Exp A':>8} {'Exp B':>8} {'Δ':>8}  {'Tendencia'}")
print('=' * 75)

for key, label in METRIC_LABELS.items():
    va    = m_a[key]
    vb    = m_b[key]
    delta = vb - va
    arrow = '↑' if delta > 0.001 else ('↓' if delta < -0.001 else '=')
    sep   = '***' if abs(delta) >= 0.02 else ('*' if abs(delta) >= 0.005 else '')
    print(f"  {label:<33} {va:>8.4f} {vb:>8.4f} {delta:>+8.4f}  {arrow} {sep}")

print('=' * 75)
print('  Δ = Exp B − Exp A   |   *** cambio significativo (≥0.02)   * cambio leve')

In [ ]:
# ── Gráfico de barras agrupadas: Exp A vs Exp B ───────────────────────────────
metric_keys  = list(METRIC_LABELS.keys())
metric_short = ['pipe\ncompleteness', 'ground\ncontinuity', 'gap\ntraversability',
                'enemy\nplacement', 'structural\navg']

vals_a = [m_a[k] for k in metric_keys]
vals_b = [m_b[k] for k in metric_keys]

x = np.arange(len(metric_keys))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars_a = ax.bar(x - w/2, vals_a, w, label='Exp A (173 fragmentos)',  color='#3498db', edgecolor='#222', linewidth=0.7)
bars_b = ax.bar(x + w/2, vals_b, w, label='Exp B (2518 ventanas)',   color='#e67e22', edgecolor='#222', linewidth=0.7)

ax.axhline(0.70, color='#888', linestyle='--', linewidth=1.2, label='Umbral objetivo (0.70)')
ax.set_xticks(x)
ax.set_xticklabels(metric_short, fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score (0–1)')
ax.set_title('Comparación de métricas estructurales — Exp A vs Exp B  (n=100)', fontsize=12, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.4)

for bar, val in zip(bars_a, vals_a):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=7, color='#3498db')
for bar, val in zip(bars_b, vals_b):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=7, color='#c0392b')

plt.tight_layout()
plt.show()

# ── Resumen numérico ──────────────────────────────────────────────────────────
print(f"\nPromedio estructural — Exp A: {m_a['structural_avg']:.4f}")
print(f"Promedio estructural — Exp B: {m_b['structural_avg']:.4f}")
delta_avg = m_b['structural_avg'] - m_a['structural_avg']
print(f"Diferencia neta             : {delta_avg:+.4f}  ({delta_avg*100:+.2f} puntos porcentuales)")

## 6. Análisis: ¿por qué más datos solos no mejoran (suficientemente)?

### Resultados observados

| Métrica | Exp A | Exp B | Δ | Interpretación |
|---|---|---|---|---|
| `pipe_completeness`  | 0.2075 | 0.1707 | −0.037 | Empeoró ligeramente |
| `ground_continuity`  | 0.5093 | 0.5211 | +0.012 | Leve mejora |
| `gap_traversability` | 0.9100 | 0.9175 | +0.008 | Estable / ligera mejora |
| `enemy_placement`    | 0.9667 | 0.9600 | −0.007 | Estable |
| `structural_avg`     | 0.6484 | 0.6423 | −0.006 | Sin mejora neta |

El **promedio estructural no mejoró**. ¿Por qué?

### Causas raíz

**1. El problema no es la cantidad de datos, sino la función de pérdida**  
La WGAN-GP minimiza la distancia de Wasserstein entre distribuciones.  
Esto equivale a replicar la distribución estadística de los tiles de entrenamiento,  
**no** a maximizar métricas de jugabilidad o coherencia estructural.  
Aunque con 14.5× más datos el modelo aprende mejor la distribución, sigue sin ningún  
incentivo para generar tuberías completas.

**2. Sobrerepresentación del tile "Vacío" (ID=2)**  
Con el sliding window de stride=4, los fragmentos capturados pueden tener mayor  
proporción de espacio vacío (secciones de cielo), desbalanceando la distribución  
hacia tiles vacíos y diluyendo la señal de aprendizaje para tiles raros (tuberías,  
enemigos).

**3. Redundancia por ventanas solapadas**  
Con stride=4 en ventanas de 28 columnas, cada columna aparece en ~7 ventanas distintas.  
Esto introduce correlación y sesgo en los mini-batches: el generador ve las mismas  
estructuras locales repetidas sin nueva información semántica.

**4. Las métricas que mejorarían requieren coherencia global**  
`pipe_completeness` depende de que TL, TR, L, R estén en las posiciones relativas  
correctas. El DCGAN genera tile a tile desde el espacio latente; sin una restricción  
explícita de layout, la expansión de datos sola no fuerza este patrón.

### Conclusión

La expansión del dataset es un paso necesario pero **no suficiente**.  
Para superar el umbral de calidad estructural se necesita:

- **Función fitness explícita** que penalice tuberías incompletas y suelo fragmentado.
- **Optimización en el espacio latente** (CMA-ES) que guíe al generador hacia regiones  
  de alta calidad estructural sin reentrenar la red.

Esta será la estrategia del **Experimento C** del proyecto NeuralPlumber.